# Báo cáo kỹ thuật: MyWeather Application

Dự án **MyWeather Application** là ứng dụng tra cứu thời tiết xây dựng bằng Streamlit, sử dụng dữ liệu từ hệ thống OpenWeather API. Ứng dụng hỗ trợ tra cứu theo tọa độ hoặc tên thành phố, hiển thị thời tiết hiện tại, chất lượng không khí, bản đồ thời tiết, dự báo 5 ngày / 3 giờ và cho phép người dùng đăng nhập để lưu địa điểm yêu thích.

**Thành phần chính:**
- `main_app.py`: giao diện Streamlit, quản lý tài khoản, sidebar, bản đồ, biểu đồ và luồng tương tác.
- `api_modules.py`: các hàm gọi OpenWeather API và chuẩn hóa dữ liệu trả về.
- `test_api_modules.py`: kiểm thử cơ bản cho các hàm API.
- `requirements.txt`: danh sách thư viện phụ thuộc.
- `.env`: lưu API key cục bộ, không đưa lên git.
- `users.json`: lưu tài khoản demo và danh sách địa điểm yêu thích.

## 1. Mô tả mục tiêu

Mục tiêu của dự án là xây dựng một ứng dụng web nhỏ phục vụ việc luyện tập gọi API, xử lý dữ liệu JSON và trình bày dữ liệu thời tiết trực quan. Ứng dụng cần đáp ứng các yêu cầu tối thiểu của bài OpenWeather API Project và bổ sung một số chức năng nâng cao để cải thiện trải nghiệm người dùng.

**Mục tiêu chức năng:**
- Cho phép người dùng tra cứu bằng **tọa độ** hoặc **tên thành phố**.
- Tự động xác định tên thành phố, quốc gia và tọa độ tương ứng.
- Hiển thị thời tiết hiện tại gồm nhiệt độ, độ ẩm, sức gió, mô tả và icon thời tiết.
- Hiển thị chất lượng không khí thông qua chỉ số AQI.
- Render bản đồ thời tiết có lớp dữ liệu OpenWeather như nhiệt độ, gió, mây và lượng mưa.
- Hiển thị biểu đồ dự báo 5 ngày / 3 giờ cho nhiệt độ, độ ẩm và sức gió.
- Hỗ trợ đăng ký, đăng nhập, lưu và xóa địa điểm yêu thích.

**Mục tiêu kỹ thuật:**
- Tách phần gọi API khỏi phần giao diện để code dễ bảo trì.
- Dùng `.env` để cấu hình API key an toàn hơn.
- Dùng `requirements.txt` để tái tạo môi trường chạy.
- Có kiểm thử cơ bản cho các luồng API chính.

## 2. Phân tích yêu cầu

| Nhóm yêu cầu | Nội dung | Trạng thái đáp ứng | File / vị trí triển khai |
|---|---|---|---|
| Core | Trích xuất tên thành phố và mã quốc gia từ tọa độ | Đã đáp ứng | `get_location_info()` trong `api_modules.py` |
| Core | Truy xuất thời tiết hiện tại | Đã đáp ứng | `get_current_weather()` trong `api_modules.py` |
| Core | Render bản đồ thời tiết có layer chuyên dụng | Đã đáp ứng | `create_weather_map()` và `render_map()` trong `main_app.py` |
| Core | Hiển thị chỉ số AQI | Đã đáp ứng | `get_air_pollution()` và `render_air_quality()` |
| Advanced | Weather widgets | Đã đáp ứng | Khối widget HTML/CSS trong `render_current_weather()` |
| Advanced | Phân tích dự báo 5 ngày / 3 giờ bằng biểu đồ | Đã đáp ứng | `get_weather_forecast()` và `render_forecast()` |
| Advanced | Quản lý người dùng và lưu thành phố quan tâm | Đã đáp ứng | `load_users()`, `save_users()`, `render_sidebar()`, `add_favorite()` |
| Submission | Notebook báo cáo kỹ thuật | Đã đáp ứng trong file này | `report.ipynb` |
| Submission | Mã nguồn `.py` tách module | Đã đáp ứng | `main_app.py`, `api_modules.py` |
| Submission | Môi trường `requirements.txt` | Đã đáp ứng | `requirements.txt` |

Kết luận: dự án hiện đã đáp ứng các yêu cầu trong README, bao gồm cả các chức năng nâng cao.

## 3. Kiến trúc tổng quát

Ứng dụng được tổ chức theo hướng tách lớp đơn giản:

- **Presentation Layer:** `main_app.py`, dùng Streamlit để tạo sidebar, layout, biểu đồ, bản đồ và các nút tương tác.
- **Service / API Layer:** `api_modules.py`, chứa các hàm gọi OpenWeather API và trả dữ liệu đã chuẩn hóa dạng dictionary.
- **Local Storage Layer:** `users.json`, lưu thông tin tài khoản demo và danh sách địa điểm yêu thích.
- **Configuration Layer:** `.env`, lưu biến môi trường `OPENWEATHER_API_KEY`.
- **Testing Layer:** `test_api_modules.py`, kiểm thử các hàm API chính.

### Sơ đồ pipeline

```mermaid
flowchart TD
    A[Người dùng nhập dữ liệu trong Sidebar] --> B{Chế độ tra cứu}
    B -->|Tọa độ| C[Nhận Latitude / Longitude]
    B -->|Tên thành phố| D[Direct Geocoding API]
    D --> C
    C --> E[Reverse Geocoding API]
    C --> F[Current Weather API]
    C --> G[Air Pollution API]
    C --> H[Weather Map Tile API]
    C --> I[5 Day / 3 Hour Forecast API]
    E --> J[Thông tin khu vực]
    F --> K[Metrics + Weather Widgets]
    G --> L[AQI]
    H --> M[Folium Map trong Streamlit]
    I --> N[Plotly Forecast Charts]
    J --> O[Dashboard kết quả]
    K --> O
    L --> O
    M --> O
    N --> O
    O --> P{Người dùng đăng nhập?}
    P -->|Có| Q[Lưu / xóa địa điểm yêu thích trong users.json]
    P -->|Không| R[Chỉ xem kết quả]
```

Luồng chính của hệ thống luôn quy mọi đầu vào về tọa độ `lat/lon`. Nhờ vậy các API còn lại có thể dùng chung một pipeline xử lý, dù người dùng nhập tọa độ trực tiếp hay nhập tên thành phố.

## 4. Quá trình thiết lập và cấu hình API

Dự án sử dụng OpenWeather API key thông qua biến môi trường để tránh hard-code khóa trong mã nguồn.

**Các bước thiết lập:**

1. Tạo tài khoản tại OpenWeather và lấy API key.
2. Tạo file `.env` tại thư mục gốc dự án.
3. Thêm biến môi trường:

```bash
OPENWEATHER_API_KEY=your_api_key_here
```

4. Trong `api_modules.py`, dùng `python-dotenv` để nạp API key:

```python
load_dotenv(dotenv_path=Path(__file__).resolve().parent / ".env")
API_KEY = os.getenv("OPENWEATHER_API_KEY")
```

**Các API endpoint được sử dụng:**
- Direct Geocoding API: đổi tên thành phố thành tọa độ.
- Reverse Geocoding API: đổi tọa độ thành tên thành phố và mã quốc gia.
- Current Weather API: lấy thời tiết hiện tại.
- Air Pollution API: lấy AQI.
- Weather Map Tile API: lấy layer bản đồ thời tiết.
- 5 Day / 3 Hour Forecast API: lấy dữ liệu dự báo cho biểu đồ.

In [ ]:
# Kiểm tra nhanh API key đã được nạp hay chưa, không in trực tiếp khóa.
from api_modules import API_KEY

print("API key configured:", bool(API_KEY))
print("API key length:", len(API_KEY) if API_KEY else 0)

## 5. Thiết kế server và giao diện ứng dụng

Ứng dụng chạy bằng Streamlit, vì vậy phần server được quản lý bởi Streamlit runtime. File khởi chạy là `main_app.py`.

**Thiết kế giao diện:**
- Sidebar gồm 3 nhóm: tài khoản, yêu thích và tra cứu.
- Form tra cứu hỗ trợ 2 chế độ: tọa độ hoặc tên thành phố.
- Kết quả tra cứu được lưu trong `st.session_state["last_search"]`, giúp dữ liệu không biến mất sau khi bấm nút lưu yêu thích hoặc khi Streamlit rerun.
- Khu vực kết quả chính chia thành 2 cột: thông tin thời tiết bên trái và bản đồ bên phải.
- Phần dự báo 5 ngày nằm riêng bên dưới bằng các tab biểu đồ.

**Thiết kế quản lý người dùng:**
- `users.json` lưu tài khoản và danh sách `favorites`.
- Người dùng có thể đăng ký, đăng nhập, đăng xuất.
- Sau khi đăng nhập, người dùng có thể lưu địa điểm đang tra cứu vào danh sách yêu thích.
- Sidebar cho phép tra cứu nhanh hoặc xóa địa điểm đã lưu.

**Lưu ý bảo mật:**
Cơ chế tài khoản hiện phù hợp với phạm vi bài tập demo. Mật khẩu đang được lưu dạng plain text trong `users.json`, chưa phù hợp cho môi trường production. Nếu triển khai thật cần hash mật khẩu, kiểm soát session và dùng database an toàn hơn.

## 6. Hướng dẫn chạy ứng dụng

Môi trường khuyến nghị là Python 3.10 trong micromamba.

```bash
# Tạo môi trường
micromamba create -n weather_env python=3.10 -c conda-forge

# Kích hoạt môi trường
micromamba activate weather_env

# Cài dependencies
pip install -r requirements.txt

# Chạy ứng dụng
streamlit run main_app.py
```

Sau khi chạy, Streamlit sẽ cung cấp URL local, thường là:

```text
http://localhost:8501
```

Người dùng mở URL này trong trình duyệt để sử dụng ứng dụng.

In [ ]:
# Có thể chạy lệnh này trong terminal thay vì trong notebook:
# !streamlit run main_app.py

## 7. Kiểm thử cơ bản hoạt động

Dự án có file `test_api_modules.py` dùng `unittest` để kiểm tra các luồng API chính.

**Các nhóm test:**
- Tọa độ hợp lệ trả về thông tin vị trí.
- Tọa độ hợp lệ trả về thời tiết hiện tại.
- Tọa độ hợp lệ trả về AQI.
- Tên thành phố hợp lệ trả về tọa độ.
- Tên thành phố không tồn tại trả về lỗi phù hợp.
- Chuỗi thành phố rỗng trả về lỗi nhập liệu.

Lệnh chạy test:

```bash
python -m unittest test_api_modules.py
```

Trong quá trình kiểm tra gần nhất, toàn bộ 6 test đều pass.

In [ ]:
# Chạy unit test từ notebook nếu đang mở trong đúng môi trường weather_env.
!python -m unittest test_api_modules.py

## 8. Các chức năng mở rộng

### 8.1 Tra cứu bằng tọa độ hoặc tên thành phố

Sidebar có radio cho phép chọn một trong hai chế độ:
- **Tọa độ:** người dùng nhập latitude và longitude.
- **Tên thành phố:** người dùng nhập tên như `Hanoi`, `Da Nang`, `Tokyo`. Ứng dụng gọi Direct Geocoding API để đổi tên thành phố thành tọa độ.

Cả hai chế độ đều quy về cùng pipeline `lat/lon`, giúp tránh lặp logic xử lý thời tiết, AQI, bản đồ và dự báo.

### 8.2 Weather Widgets

Ứng dụng hiển thị thêm các thông số chi tiết dưới dạng widget:
- Cảm giác như.
- Tầm nhìn.
- Áp suất.
- Thời gian mặt trời mọc và lặn.

### 8.3 Biểu đồ dự báo 5 ngày / 3 giờ

Dữ liệu dự báo từ Forecast API được chuyển thành Pandas DataFrame, sau đó hiển thị bằng Plotly trong 3 tab:
- Nhiệt độ.
- Độ ẩm.
- Sức gió.

### 8.4 Quản lý người dùng và địa điểm yêu thích

Người dùng có thể:
- Đăng ký tài khoản.
- Đăng nhập / đăng xuất.
- Lưu địa điểm đang tra cứu vào danh sách yêu thích.
- Tra cứu nhanh địa điểm đã lưu.
- Xóa địa điểm khỏi danh sách yêu thích.

Dữ liệu demo được lưu trong `users.json`.

## 9. Đánh giá mức độ đáp ứng README

Dựa trên mã nguồn hiện tại, dự án đã đáp ứng đầy đủ các yêu cầu trong `README.md`.

**Core Features:** 4/4
- Định vị không gian: đạt.
- Thời tiết thời gian thực: đạt.
- Bản đồ thời tiết có layer: đạt.
- AQI: đạt.

**Advanced Features:** 3/3
- Weather widgets: đạt.
- Forecast chart 5 ngày / 3 giờ: đạt.
- Quản lý người dùng và thành phố quan tâm: đạt.

**Submission Structure:** 3/3
- Có notebook báo cáo kỹ thuật: đạt.
- Có mã nguồn `.py` tách module: đạt.
- Có `requirements.txt`: đạt.

Tổng quan: dự án đạt khoảng **100% yêu cầu README** ở mức bài tập/lab.

## 10. Hạn chế và hướng phát triển

Một số điểm có thể cải thiện nếu phát triển tiếp:
- Hash mật khẩu thay vì lưu plain text.
- Chuyển `users.json` sang SQLite hoặc database thật.
- Thêm cache cho API để giảm số lần gọi khi người dùng tra cứu lặp lại.
- Thêm lựa chọn đơn vị nhiệt độ Celsius / Fahrenheit.
- Thêm xử lý timezone theo địa điểm thay vì dùng timezone của máy chạy server.
- Bổ sung kiểm thử giao diện Streamlit hoặc mock API để test không phụ thuộc mạng.